# Demo Dataset Explorer

Visual inspection and validation of HDF5 datasets produced by `collect_demos.py`.

**Dataset layout reminder:**
```
episode_N/
    obs/
        camera_rgb       (T, H, W, 3)   uint8
        base_pose        (T, 7)         float32  pos(3) + quat_wxyz(4)
        base_lin_vel     (T, 3)         float32  world frame
        base_ang_vel     (T, 3)         float32  world frame
        joint_pos        (T, 12)        float32
        joint_vel        (T, 12)        float32
        contact_forces   (T, N, 3)      float32  net forces world frame
        goal_relative    (T, 3)         float32  goal minus robot XYZ
        policy_obs       (T, 48)        float32  raw locomotion policy input
    action/
        velocity_cmd     (T, 3)         float32  [v_x, v_y, omega_z]
        joint_targets    (T, 12)        float32  raw policy output
```

In [ ]:
import glob
import os

import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display
import ipywidgets as widgets

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# ── File selection ────────────────────────────────────────────────────────────
DATASETS_DIR = os.path.join("..", "datasets")
hdf5_files = sorted(glob.glob(os.path.join(DATASETS_DIR, "*.hdf5")))

assert hdf5_files, f"No .hdf5 files found in {DATASETS_DIR}"

# Change this to point at a different file
HDF5_PATH = hdf5_files[-1]   # latest by default

print(f"Available datasets:")
for i, p in enumerate(hdf5_files):
    marker = " ← selected" if p == HDF5_PATH else ""
    print(f"  [{i}] {os.path.basename(p)}{marker}")

In [ ]:
# ── Load all episodes into memory ─────────────────────────────────────────────
episodes = {}

with h5py.File(HDF5_PATH, "r") as f:
    session_ts = f.attrs.get("session_timestamp", "unknown")
    for key in sorted(f.keys()):
        ep = f[key]
        episodes[key] = {
            # attributes
            "success":            bool(ep.attrs["success"]),
            "episode_length":     int(ep.attrs["episode_length"]),
            "goal_position_world": ep.attrs["goal_position_world"][:],
            "timestamp":          ep.attrs.get("timestamp", ""),
            # arrays
            "camera_rgb":         ep["obs/camera_rgb"][()],
            "base_pose":          ep["obs/base_pose"][()],
            "base_lin_vel":       ep["obs/base_lin_vel"][()],
            "base_ang_vel":       ep["obs/base_ang_vel"][()],
            "joint_pos":          ep["obs/joint_pos"][()],
            "joint_vel":          ep["obs/joint_vel"][()],
            "contact_forces":     ep["obs/contact_forces"][()],
            "goal_relative":      ep["obs/goal_relative"][()],
            "policy_obs":         ep["obs/policy_obs"][()],
            "velocity_cmd":       ep["action/velocity_cmd"][()],
            "joint_targets":      ep["action/joint_targets"][()],
        }

ep_keys  = list(episodes.keys())
n_ep     = len(ep_keys)
n_success = sum(1 for e in episodes.values() if e["success"])

print(f"Session : {session_ts}")
print(f"File    : {os.path.basename(HDF5_PATH)}")
print(f"Episodes: {n_ep}  ({n_success} success / {n_ep - n_success} failure)")
print()
for k, e in episodes.items():
    print(f"  {k:12s}  T={e['episode_length']:4d}  success={e['success']}  "
          f"goal={e['goal_position_world']}  @ {e['timestamp']}")

---
## 1 · Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Dataset overview — {os.path.basename(HDF5_PATH)}", fontsize=11)

# ── Episode lengths bar chart ─────────────────────────────────────────────────
ax = axes[0]
lengths = [e["episode_length"] for e in episodes.values()]
colors  = ["tab:green" if e["success"] else "tab:red" for e in episodes.values()]
ax.bar(ep_keys, lengths, color=colors, edgecolor="white", linewidth=0.6)
ax.axhline(np.mean(lengths), color="black", linestyle="--", linewidth=1, label=f"mean={np.mean(lengths):.0f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps (10 Hz)")
ax.set_title("Episode lengths")
ax.tick_params(axis="x", rotation=45)
ax.legend(handles=[
    Patch(color="tab:green", label="success"),
    Patch(color="tab:red",   label="failure"),
], loc="upper right")
ax.legend(loc="upper right")

# ── Velocity command distribution (all episodes combined) ─────────────────────
ax = axes[1]
all_cmds = np.concatenate([e["velocity_cmd"] for e in episodes.values()], axis=0)
labels  = ["v_x (m/s)", "v_y (m/s)", "ω_z (rad/s)"]
for i, label in enumerate(labels):
    ax.hist(all_cmds[:, i], bins=40, alpha=0.6, label=label)
ax.set_xlabel("Commanded value")
ax.set_ylabel("Count")
ax.set_title("Velocity command distribution (all episodes)")
ax.legend()

plt.tight_layout()
plt.show()

---
## 2 · Camera RGB Viewer

Use the slider to scrub through frames, or pick a different episode from the dropdown.

In [ ]:
ep_selector = widgets.Dropdown(
    options=ep_keys,
    value=ep_keys[0],
    description="Episode:",
)
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=episodes[ep_keys[0]]["episode_length"] - 1,
    step=1,
    description="Frame:",
    continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
out_img = widgets.Output()

def update_frame(*_):
    ep  = episodes[ep_selector.value]
    t   = frame_slider.value
    rgb = ep["camera_rgb"][t]          # (H, W, 3)
    vc  = ep["velocity_cmd"][t]        # (3,)
    pos = ep["base_pose"][t, :3]       # (3,)
    frame_slider.max = ep["episode_length"] - 1
    with out_img:
        out_img.clear_output(wait=True)
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        ax.imshow(rgb)
        ax.set_title(
            f"{ep_selector.value}  frame {t}/{ep['episode_length']-1}  "
            f"success={ep['success']}\n"
            f"vx={vc[0]:+.2f}  vy={vc[1]:+.2f}  wz={vc[2]:+.2f}  "
            f"pos=({pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f})",
            fontsize=8,
        )
        ax.axis("off")
        plt.tight_layout()
        plt.show()

def on_episode_change(change):
    ep = episodes[change["new"]]
    frame_slider.max = ep["episode_length"] - 1
    frame_slider.value = 0
    update_frame()

ep_selector.observe(on_episode_change, names="value")
frame_slider.observe(update_frame, names="value")

display(widgets.VBox([ep_selector, frame_slider, out_img]))
update_frame()

---
## 3 · XY Trajectory (Bird's-Eye View)

Top-down view of Spot's path in the warehouse for every episode.
The goal marker is shown as a green star; start/end positions are marked.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cmap = plt.get_cmap("tab10")

for i, (key, ep) in enumerate(episodes.items()):
    xy   = ep["base_pose"][:, :2]        # (T, 2)
    col  = cmap(i % 10)
    ls   = "-" if ep["success"] else "--"
    ax.plot(xy[:, 0], xy[:, 1], linestyle=ls, color=col, linewidth=1.5,
            label=f"{key} ({'S' if ep['success'] else 'F'}, T={ep['episode_length']})")
    # Start dot
    ax.scatter(xy[0, 0],  xy[0, 1],  marker="o", s=60,  color=col, zorder=5)
    # End arrow
    ax.annotate("", xy=(xy[-1, 0], xy[-1, 1]),
                xytext=(xy[-2, 0], xy[-2, 1]),
                arrowprops=dict(arrowstyle="->", color=col, lw=1.5))

# Goal marker (same for all episodes in this session)
goal = episodes[ep_keys[0]]["goal_position_world"]
ax.scatter(goal[0], goal[1], marker="*", s=300, color="limegreen",
           edgecolors="black", linewidths=0.8, zorder=6, label=f"Goal ({goal[0]:.1f}, {goal[1]:.1f})")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Spot XY trajectories (solid=success, dashed=failure)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

---
## 4 · Velocity Commands vs Actual Base Velocity

Compares what you *commanded* (gamepad) with what the robot *actually achieved*.
Lag and saturation indicate how well the locomotion policy is tracking the command.

In [ ]:
# Training velocity limits (shaded bands)
VEL_LIMITS = [(-2.0, 3.0), (-1.5, 1.5), (-2.0, 2.0)]
VEL_LABELS = ["v_x (m/s)", "v_y (m/s)", "ω_z (rad/s)"]

ep_dropdown = widgets.Dropdown(options=ep_keys, value=ep_keys[0], description="Episode:")
out_vel = widgets.Output()

def plot_velocity(ep_key):
    ep  = episodes[ep_key]
    T   = ep["episode_length"]
    t   = np.arange(T) / 10.0        # seconds at 10 Hz
    cmd = ep["velocity_cmd"]          # (T, 3)
    actual_lin = ep["base_lin_vel"]   # (T, 3)  world frame: x, y, z
    actual_ang = ep["base_ang_vel"]   # (T, 3)  world frame: roll_rate, pitch_rate, yaw_rate

    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    fig.suptitle(f"{ep_key} — velocity commanded vs actual  (success={ep['success']})", fontsize=10)

    actual_signals = [actual_lin[:, 0], actual_lin[:, 1], actual_ang[:, 2]]

    for i, ax in enumerate(axes):
        lo, hi = VEL_LIMITS[i]
        ax.axhspan(lo, hi, alpha=0.08, color="steelblue", label="training range")
        ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
        ax.plot(t, cmd[:, i],           color="tab:orange", linewidth=1.5, label="commanded")
        ax.plot(t, actual_signals[i],   color="tab:blue",   linewidth=1.0, alpha=0.8, label="actual")
        ax.set_ylabel(VEL_LABELS[i])
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

def on_vel_ep_change(change):
    with out_vel:
        out_vel.clear_output(wait=True)
        plot_velocity(change["new"])

ep_dropdown.observe(on_vel_ep_change, names="value")
display(ep_dropdown, out_vel)

with out_vel:
    plot_velocity(ep_keys[0])

---
## 5 · Policy-Obs / Velocity-Cmd Consistency Check

`policy_obs[9:12]` must equal `velocity_cmd` exactly — they are set from the same
tensor before policy inference. Any non-zero residual indicates a data-logging bug.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("policy_obs[9:12] vs velocity_cmd  (should lie exactly on y = x)", fontsize=10)

for ep_key, ep in episodes.items():
    obs_slice = ep["policy_obs"][:, 9:12]   # (T, 3)
    vel_cmd   = ep["velocity_cmd"]            # (T, 3)
    for i, ax in enumerate(axes):
        ax.scatter(vel_cmd[:, i], obs_slice[:, i], s=4, alpha=0.5, label=ep_key)

axis_labels = ["v_x (m/s)", "v_y (m/s)", "ω_z (rad/s)"]
all_vals = np.concatenate([e["velocity_cmd"] for e in episodes.values()], axis=0)

for i, ax in enumerate(axes):
    lim = [all_vals[:, i].min() - 0.1, all_vals[:, i].max() + 0.1]
    ax.plot(lim, lim, "r--", linewidth=1, label="y = x (ideal)")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel(f"velocity_cmd  {axis_labels[i]}")
    ax.set_ylabel(f"policy_obs[9+{i}]")
    ax.set_title(axis_labels[i])
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

axes[-1].legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

# Numeric residual
print("Max |policy_obs[9:12] − velocity_cmd| per episode:")
for ep_key, ep in episodes.items():
    diff = np.abs(ep["policy_obs"][:, 9:12] - ep["velocity_cmd"]).max()
    status = "✓" if diff < 1e-5 else "✗ MISMATCH"
    print(f"  {ep_key}: max diff = {diff:.2e}  {status}")

---
## 6 · Joint Positions Heatmap

Each row is one of the 12 Spot joints (3 per leg × 4 legs).
Large deviations from default pose indicate dynamic locomotion or falls.

In [ ]:
# Approximate Spot joint names in Isaac Lab order
# FL = front-left, FR = front-right, RL = rear-left, RR = rear-right
# HX = hip-abduction, HY = hip-flexion, KN = knee
JOINT_NAMES = [
    "FL_hx", "FL_hy", "FL_kn",
    "FR_hx", "FR_hy", "FR_kn",
    "RL_hx", "RL_hy", "RL_kn",
    "RR_hx", "RR_hy", "RR_kn",
]

ep_sel2 = widgets.Dropdown(options=ep_keys, value=ep_keys[0], description="Episode:")
out_jnt  = widgets.Output()

def plot_joints(ep_key):
    ep  = episodes[ep_key]
    T   = ep["episode_length"]
    t   = np.arange(T) / 10.0
    jp  = ep["joint_pos"]    # (T, 12)
    jt  = ep["joint_targets"] # (T, 12)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{ep_key} — joint positions & targets  (success={ep['success']})", fontsize=10)

    for ax, data, title in zip(axes, [jp, jt], ["joint_pos (measured)", "joint_targets (policy output)"]):
        im = ax.imshow(
            data.T, aspect="auto", origin="lower",
            extent=[t[0], t[-1], -0.5, 11.5],
            cmap="RdBu_r", vmin=-2.5, vmax=2.5,
        )
        ax.set_yticks(np.arange(12))
        ax.set_yticklabels(JOINT_NAMES, fontsize=8)
        ax.set_xlabel("Time (s)")
        ax.set_title(title)
        plt.colorbar(im, ax=ax, label="rad")

    plt.tight_layout()
    plt.show()

def on_jnt_change(change):
    with out_jnt:
        out_jnt.clear_output(wait=True)
        plot_joints(change["new"])

ep_sel2.observe(on_jnt_change, names="value")
display(ep_sel2, out_jnt)

with out_jnt:
    plot_joints(ep_keys[0])

---
## 7 · Contact Force Magnitudes

Net force magnitude (||F||) for every body in the contact sensor.
Foot bodies should show periodic loading (gait), while non-foot bodies should be near zero unless Spot fell.

In [ ]:
ep_sel3  = widgets.Dropdown(options=ep_keys, value=ep_keys[0], description="Episode:")
out_cf   = widgets.Output()

def plot_contact_forces(ep_key):
    ep  = episodes[ep_key]
    T   = ep["episode_length"]
    t   = np.arange(T) / 10.0
    cf  = ep["contact_forces"]              # (T, N_bodies, 3)
    mag = np.linalg.norm(cf, axis=-1)       # (T, N_bodies)
    N   = mag.shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{ep_key} — contact forces  (success={ep['success']})", fontsize=10)

    # ── Heatmap ───────────────────────────────────────────────────────────────
    ax = axes[0]
    im = ax.imshow(
        mag.T, aspect="auto", origin="lower",
        extent=[t[0], t[-1], -0.5, N - 0.5],
        cmap="hot_r", vmin=0, vmax=mag.max(),
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Body index")
    ax.set_title("Force magnitude heatmap (N)")
    plt.colorbar(im, ax=ax, label="||F|| (N)")

    # ── Top 4 highest-contact bodies (likely feet) ────────────────────────────
    ax = axes[1]
    mean_force = mag.mean(axis=0)           # (N_bodies,)
    top4 = np.argsort(mean_force)[-4:][::-1]
    for body_idx in top4:
        ax.plot(t, mag[:, body_idx], linewidth=1.2, label=f"body {body_idx} (mean={mean_force[body_idx]:.1f} N)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("||F|| (N)")
    ax.set_title("Top 4 most-loaded bodies (likely feet)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def on_cf_change(change):
    with out_cf:
        out_cf.clear_output(wait=True)
        plot_contact_forces(change["new"])

ep_sel3.observe(on_cf_change, names="value")
display(ep_sel3, out_cf)

with out_cf:
    plot_contact_forces(ep_keys[0])

---
## 8 · Frame Strip — Visual Gait Sampling

Shows 8 evenly-spaced camera frames from an episode in a single row,
with the commanded velocity annotated on each frame.

In [ ]:
N_FRAMES = 8

ep_sel4 = widgets.Dropdown(options=ep_keys, value=ep_keys[0], description="Episode:")
out_strip = widgets.Output()

def plot_strip(ep_key):
    ep   = episodes[ep_key]
    T    = ep["episode_length"]
    idxs = np.linspace(0, T - 1, N_FRAMES, dtype=int)

    fig, axes = plt.subplots(1, N_FRAMES, figsize=(16, 2.5))
    fig.suptitle(
        f"{ep_key} — {N_FRAMES} evenly-spaced frames  "
        f"(T={T}, success={ep['success']})",
        fontsize=9,
    )
    for ax, t_idx in zip(axes, idxs):
        ax.imshow(ep["camera_rgb"][t_idx])
        vc = ep["velocity_cmd"][t_idx]
        ax.set_title(
            f"t={t_idx/10:.1f}s\nvx={vc[0]:+.1f} vy={vc[1]:+.1f}\nwz={vc[2]:+.1f}",
            fontsize=6,
        )
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def on_strip_change(change):
    with out_strip:
        out_strip.clear_output(wait=True)
        plot_strip(change["new"])

ep_sel4.observe(on_strip_change, names="value")
display(ep_sel4, out_strip)

with out_strip:
    plot_strip(ep_keys[0])

---
## 9 · Goal-Relative Distance Over Time

Euclidean XY distance from Spot to the goal throughout each episode.
A successful episode should end near `success_radius = 0.5 m`.

In [ ]:
SUCCESS_RADIUS = 0.5   # metres, must match collect_demos.py --success_radius

fig, ax = plt.subplots(figsize=(12, 4))
cmap = plt.get_cmap("tab10")

for i, (key, ep) in enumerate(episodes.items()):
    T   = ep["episode_length"]
    t   = np.arange(T) / 10.0
    gr  = ep["goal_relative"]                   # (T, 3)
    dist_xy = np.linalg.norm(gr[:, :2], axis=-1) # XY only
    col = cmap(i % 10)
    ls  = "-" if ep["success"] else "--"
    ax.plot(t, dist_xy, color=col, linestyle=ls, linewidth=1.5,
            label=f"{key} ({'S' if ep['success'] else 'F'})")

ax.axhline(SUCCESS_RADIUS, color="limegreen", linestyle=":", linewidth=1.5,
           label=f"success radius ({SUCCESS_RADIUS} m)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance to goal (m)")
ax.set_title("XY distance to goal over time  (solid=success, dashed=failure)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()